# Sentiment Reputation Monitoring

Progetto MLops sviluppato per il monitoraggio
della reputazione online tramite analisi del sentiment.

**Contenuti dimostrati:**
- Model serving con FastAPI
- Test automatici (CI)
- Deploy su Hugging Face Space (CD)
- Monitoring (metriche + log)
- Data drift (controllo semplice)

**Repository GitHub:** https://github.com/albertoliuzzo/MLops-sentiment-reputation  
**API live (Hugging Face Space):** https://AlbertoLiuzzo-mlops-sentiment-reputation.hf.space


In [1]:
#Clona il repository per riproducibilità
!git clone https://github.com/albertoliuzzo/MLOps-sentiment-reputation.git
%cd MLOps-sentiment-reputation

Cloning into 'MLOps-sentiment-reputation'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 88 (delta 29), reused 75 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 15.56 KiB | 3.11 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/MLOps-sentiment-reputation


In [2]:
!ls

app  Dockerfile  monitoring  README.md	requirements.txt  scripts  tests


In [13]:
#scarico il dataset pubblico tweet_eval di HF, prendo 100 esempi del test set e mappo le label
!pip -q install datasets
from datasets import load_dataset

ds = load_dataset("tweet_eval", "sentiment")
test_set = ds["test"].to_pandas().head(100)

label_map = {0: "negative", 1: "neutral", 2: "positive"}

test_set["sentiment"] = test_set["label"].map(label_map)

In [15]:
test_set.head(5), test_set["sentiment"].value_counts()

(                                                text  label sentiment
 0  @user @user what do these '1/2 naked pics' hav...      1   neutral
 1  OH: “I had a blue penis while I was this” [pla...      1   neutral
 2  @user @user That's coming, but I think the vic...      1   neutral
 3  I think I may be finally in with the in crowd ...      2  positive
 4  @user Wow,first Hugo Chavez and now Fidel Cast...      0  negative,
 sentiment
 neutral     55
 negative    29
 positive    16
 Name: count, dtype: int64)

In [16]:
#Installa dipendenze
!pip install -r requirements.txt

In [17]:
#Esegue i test
!pytest -q

....                                                                     [100%]
4 passed in 40.63s


In [36]:
#Verifica che API è live (health check) e endpoint di monitoring esposto
import requests

SPACE_URL = "https://AlbertoLiuzzo-mlops-sentiment-reputation.hf.space"

# health check
r = requests.get(f"{SPACE_URL}/health", timeout=30)
print("HEALTH:", r.status_code, r.json())

# metrics endpoint up. /metrics espone metriche in formato Prometheus, utili per una dashboard Grafana (che non ho creato in questo progetto)
m = requests.get(f"{SPACE_URL}/metrics", timeout=30)
print("METRICS STATUS:", m.status_code)

HEALTH: 200 {'status': 'ok'}
METRICS STATUS: 200


In [19]:
#Chiamata di inferenza di prova
sentence = {"text": "I love this product"}
r = requests.post(f"{SPACE_URL}/predict", json=sentence)

r.json()

{'Sentiment': 'positive', 'Probabilità': 0.9794027209281921}

In [37]:
import random
import pandas as pd

#Calcolo predizioni, chiamando il modello tramite API, sui primi 30 esempi del test set
sample = test_set.sample(30, random_state=42)

results = []

for _, row in sample.iterrows():
    text = row["text"]
    true = row["sentiment"]

    r = requests.post(f"{SPACE_URL}/predict", json={"text": text}, timeout=30)
    r.raise_for_status()

    pred = r.json()["Sentiment"]
    prob = float(r.json()["Probabilità"])

    results.append({
        "text": text,
        "true": true,
        "pred": pred,
        "prob": prob,
        "text_len": len(str(text))
    })

df_results = pd.DataFrame(results)
df_results.head(5)

,text,true,pred,prob,text_len
0,Buy VAMAA 3D Printing Pen 1.75mm ABS Filament ...,neutral,neutral,0.895314,94
1,@user @user @user @user @user @user I'm gettin...,negative,negative,0.941994,91
2,@user @user @user only wants adulation and syc...,negative,negative,0.846941,105
3,Grayson Allen just gave dude such a sick move,negative,positive,0.732522,45
4,"So that Westworld show, it isn't too bad. I gu...",neutral,positive,0.823005,82


In [38]:
print(f"Ho appena generato {len(results)} chiamate a /predict.")
print("Questo simula traffico reale verso l'API (monitorabile via /metrics in un sistema Prometheus/Grafana).")

Ho appena generato 30 chiamate a /predict.
Questo simula traffico reale verso l'API (monitorabile via /metrics in un sistema Prometheus/Grafana).


In [27]:
#scrittura del log nel file predictions.jsonl
from pathlib import Path
import json

Path("monitoring").mkdir(exist_ok=True)

log_path = "monitoring/predictions.jsonl"
with open(log_path, "w", encoding="utf-8") as f:
    for r in results:
        # questo è il formato log usato dal drift check:
        # deve avere almeno "Sentiment" e qualcosa per la lunghezza (qui "text_len")
        f.write(json.dumps({
            "Sentiment": r["pred"],
            "text_len": r["text_len"],
            "Probabilità": r["prob"]
        }) + "\n")

log_path

'monitoring/predictions.jsonl'

In [29]:
#calcolo accuracy sui sample
accuracy = (df_results["true"] == df_results["pred"]).mean()
print(f"Accuracy sul sample: {accuracy:.2f}")

Accuracy sul sample: 0.73


In [30]:
#Data Drift (semplice). Controlla se la distribuzione delle predizioni loggate e la lunghezza dei testi cambiano rispetto a una baseline.
!python monitoring/drift_check_simple.py

Window size: 30
Current label dist: {'unknown': 1.0}
Current avg text_len: 82.8

DRIFT/ALERT DETECTED:
- Label shift: positive baseline=0.33 current=0.00 diff=0.33
- Label shift: neutral baseline=0.34 current=0.00 diff=0.34
- Label shift: negative baseline=0.33 current=0.00 diff=0.33


## Conclusione

Il progetto dimostra un flusso MLOps completo:
- sviluppo e serving del modello
- test automatici
- deploy continuo
- monitoraggio in produzione
- controllo del data drift
